In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.metrics import confusion_matrix, roc_auc_score, accuracy_score, classification_report

cleaned_data_path = os.path.join('f1_data_cleaned', 'f1_processed_dataset.csv')
df_f1 = pd.read_csv(cleaned_data_path)

categorical_features = ['driver_nationality', 'constructor_nationality']
numerical_features = ['grid', 'year', 'round', 'circuitId', 'driver_age', 'quali_position']

X = df_f1[categorical_features + numerical_features]
y = df_f1['top3']

# Podział 80/10/10: treningowy / walidacyjny / testowy
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.2, random_state=1, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=1, stratify=y_temp
)

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ],
    remainder='passthrough'
)

X_train_trans = preprocessor.fit_transform(X_train)
X_test_trans = preprocessor.transform(X_test)

encoded_cat_names = preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_features).tolist()
all_feature_names = encoded_cat_names + numerical_features

print(f"Preprocessing zakończony pomyślnie!")
print(f"Liczba cech wejściowych po kodowaniu One-Hot: {X_train_trans.shape[1]}")

In [ ]:
# Wariant A: Drzewo pełne
tree_wariant_a = DecisionTreeClassifier(criterion='gini', random_state=1)
tree_wariant_a.fit(X_train_trans, y_train)

# Wariant B: Drzewo optymalne
tree_wariant_b = DecisionTreeClassifier(criterion='gini', max_depth=4, min_samples_leaf=20, random_state=1)
tree_wariant_b.fit(X_train_trans, y_train)

# Wariant C: Drzewo zbalansowane (Pre-pruning + automatyczna korekta wag asymetrii klas)
tree_wariant_c = DecisionTreeClassifier(criterion='gini', max_depth=4, min_samples_leaf=20, class_weight='balanced', random_state=1)
tree_wariant_c.fit(X_train_trans, y_train)

print("Wszystkie trzy warianty drzewa zostały pomyślnie wytrenowane!")

In [ ]:
modele_drzewiaste = {
    'Wariant A (Pełne)': tree_wariant_a,
    'Wariant B (Optymalne)': tree_wariant_b,
    'Wariant C (Zbalansowane)': tree_wariant_c
}

podsumowanie_wynikow = []

for nazwa, model in modele_drzewiaste.items():
    pred_etykiety = model.predict(X_test_trans)
    pred_prawdopodobienstwa = model.predict_proba(X_test_trans)[:, 1]
    
    tn, fp, fn, tp = confusion_matrix(y_test, pred_etykiety).ravel()
    
    dokladnosc = accuracy_score(y_test, pred_etykiety)
    czulosc = tp / (tp + fn)
    specyficznosc = tn / (tn + fp)
    auc = roc_auc_score(y_test, pred_prawdopodobienstwa)
    
    podsumowanie_wynikow.append({
        'Wariant Modelu': nazwa,
        'Accuracy': round(dokladnosc, 4),
        'Czułość (Sensitivity)': round(czulosc, 4),
        'Specyficzność (Specificity)': round(specyficznosc, 4),
        'AUC Score': round(auc, 4)
    })

df_podsumowanie = pd.DataFrame(podsumowanie_wynikow)
print("\n--- TABELA ZBIORCZA METRYK DLA MODELU DRZEWIASTEGO ---")
display(df_podsumowanie)

In [ ]:
import os
os.makedirs('Plots', exist_ok=True)

nazwa_do_pliku = {
    'Wariant B (Optymalne)': 'macierz_b',
    'Wariant C (Zbalansowane)': 'macierz_c'
}

for nazwa in ['Wariant B (Optymalne)', 'Wariant C (Zbalansowane)']:
    model = modele_drzewiaste[nazwa]
    pred_etykiety = model.predict(X_test_trans)

    print('\n' + '='*60)
    print(f'RAPORT KLASYFIKACJI DLA: {nazwa}')
    print('='*60)
    print(classification_report(y_test, pred_etykiety, target_names=['Brak Podium (0)', 'Podium (1)']))

    cm = confusion_matrix(y_test, pred_etykiety)
    print(f'Macierz pomylek dla {nazwa}:')
    print(f'True Negatives (Prawidlowy brak podium): {cm[0,0]}')
    print(f'False Positives (Bledne wytypowanie podium - Blad I rodzaju): {cm[0,1]}')
    print(f'False Negatives (Pominiete podium - Blad II rodzaju): {cm[1,0]}')
    print(f'True Positives (Prawidlowo wykryte podium): {cm[1,1]}\n')

    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Brak Podium', 'Podium'],
                yticklabels=['Brak Podium', 'Podium'])
    plt.title(f'Macierz pomylek - {nazwa}')
    plt.ylabel('Rzeczywistosc')
    plt.xlabel('Predykcja')
    plt.tight_layout()
    plik = os.path.join('Plots', nazwa_do_pliku[nazwa] + '.png')
    plt.savefig(plik, dpi=150, bbox_inches='tight')
    print(f'Zapisano: {plik}')
    plt.show()


In [ ]:
plt.figure(figsize=(22, 10))
plot_tree(tree_wariant_b,
          feature_names=all_feature_names,
          class_names=['Brak Podium', 'Podium'],
          filled=True,
          rounded=True,
          fontsize=10)
plt.title('Struktura podzialow przestrzeni cech dla optymalnego drzewa (Wariant B)', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join('Plots', 'struktura_drzewa_cart.png'), dpi=150, bbox_inches='tight')
print('Zapisano: Plots/struktura_drzewa_cart.png')
plt.show()

print('\n--- TEKSTOWE REGULY DECYZYJNE (WARIANT B) ---')
print(export_text(tree_wariant_b, feature_names=all_feature_names))
